In [1]:
#Import the petting zoo environment and evaluate
from Multi_Ag_Environment import CustomEnvironment
from pettingzoo.test import parallel_api_test
import gymnasium as gym
env = CustomEnvironment()
parallel_api_test(env, num_cycles=1_000_000)

[45, 8, 52]
New Matrix
[[0.8450684714319319, 10, ['Satellite2']], [0.09166974670302819, 10, ['Satellite3']], [0.14876518092741875, 10, ['Satellite2', 'Satellite1']], [0.2992469934818275, 10, ['Satellite2', 'Satellite3', 'Satellite1']], [0.968822588042101, 10, ['Satellite1', 'Satellite2', 'Satellite3']], [0.32102041537954296, 10, ['Satellite2', 'Satellite3', 'Satellite1']], [0.046557587309588255, 10, ['Satellite1', 'Satellite3', 'Satellite2']], [0.3552284846294942, 10, ['Satellite3', 'Satellite2', 'Satellite1']], [0.37103181336229474, 10, ['Satellite1', 'Satellite2']], [0.058592580265435235, 10, ['Satellite3', 'Satellite2', 'Satellite1']], [0.7005893653418104, 10, ['Satellite3', 'Satellite2', 'Satellite1']], [0.6477706467491113, 10, ['Satellite1', 'Satellite3']], [0.9126028824731294, 10, ['Satellite3']], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1

In [ ]:
#More intelligent heuristic
import random

# Initialize the env
env = CustomEnvironment()
observations = env.reset()

# Use a fixed agent list
all_agents = ["satellite1", "satellite2", "satellite3"]

# Track done flags
terminated = {agent: False for agent in all_agents}
truncated = {agent: False for agent in all_agents}

while not all([terminated[a] or truncated[a] for a in all_agents]):
    actions = {
        agent: env.action_space(agent).sample()
        for agent in all_agents
        if not (terminated[agent] or truncated[agent])
    }
    print("Agent Actions")
    print(actions)
    observations, rewards, terminated, truncated, infos = env.step(actions)
    print("Rewards")
    print(rewards)
env.close()

In [ ]:
#CGR
#Here we introduce CGR which will always choose Action one regardless of the weather conditions
observations = env.reset()

# Use a fixed agent list
all_agents = ["satellite1", "satellite2", "satellite3"]

# Track done flags
terminated = {agent: False for agent in all_agents}
truncated = {agent: False for agent in all_agents}

while not all([terminated[a] or truncated[a] for a in all_agents]):
    actions = {
        agent:1
        for agent in all_agents
        if not (terminated[agent] or truncated[agent])
    }
    print("Agent Actions")
    print(actions)
    observations, rewards, terminated, truncated, infos = env.step(actions)
    print("Rewards")
    print(rewards)
env.close()

In [ ]:
#Heuristic Baseline
observations = env.reset()
#Next we need to conduct the initial contact selection based upon the initial state


all_agents = ["satellite1", "satellite2", "satellite3"]


terminated = {agent: False for agent in all_agents}
truncated = {agent: False for agent in all_agents}

while not all([terminated[a] or truncated[a] for a in all_agents]):
    actions = {
        agent:1
        for agent in all_agents
        if not (terminated[agent] or truncated[agent])
    }
    print("Agent Actions")
    print(actions)
    observations, rewards, terminated, truncated, infos = env.step(actions)
    print("Rewards")
    print(rewards)
env.close()

[100, 63, 65]
New Matrix
[[0.18922649762591703, 10, ['Satellite1']], [0.8185155505112031, 10, ['Satellite3']], [0.21415438501739337, 10, ['Satellite2', 'Satellite1']], [0.17607526895884484, 10, ['Satellite3', 'Satellite1', 'Satellite2']], [0.43885110298163166, 10, ['Satellite3', 'Satellite2', 'Satellite1']], [0.7832047275778536, 10, ['Satellite2', 'Satellite1', 'Satellite3']], [0.6468191860543259, 10, ['Satellite3']], [0.47458261256566425, 10, ['Satellite1']], [0.37449138903978585, 10, ['Satellite1', 'Satellite2']], [0.5938852023891397, 10, ['Satellite1', 'Satellite2']], [0.7648528981397799, 10, ['Satellite2', 'Satellite3', 'Satellite1']], [0.8614524409428448, 10, ['Satellite2', 'Satellite3', 'Satellite1']], [0.6577604452637925, 10, ['Satellite3']], [0.8546602429364465, 10, ['Satellite2', 'Satellite3']], [0.6531283535295089, 10, ['Satellite3', 'Satellite2']], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-

In [ ]:
import os

import ray
import supersuit as ss
from ray import tune
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.env.wrappers.pettingzoo_env import ParallelPettingZooEnv
from ray.rllib.models import ModelCatalog
from ray.rllib.models.torch.torch_modelv2 import TorchModelV2
from ray.tune.registry import register_env
from torch import nn

#from pettingzoo.butterfly import pistonball_v6


class CNNModelV2(TorchModelV2, nn.Module):
    def __init__(self, obs_space, act_space, num_outputs, *args, **kwargs):
        TorchModelV2.__init__(self, obs_space, act_space, num_outputs, *args, **kwargs)
        nn.Module.__init__(self)
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, [8, 8], stride=(4, 4)),
            nn.ReLU(),
            nn.Conv2d(32, 64, [4, 4], stride=(2, 2)),
            nn.ReLU(),
            nn.Conv2d(64, 64, [3, 3], stride=(1, 1)),
            nn.ReLU(),
            nn.Flatten(),
            (nn.Linear(3136, 512)),
            nn.ReLU(),
        )
        self.policy_fn = nn.Linear(512, num_outputs)
        self.value_fn = nn.Linear(512, 1)

    def forward(self, input_dict, state, seq_lens):
        model_out = self.model(input_dict["obs"].permute(0, 3, 1, 2))
        self._value_out = self.value_fn(model_out)
        return self.policy_fn(model_out), state

    def value_function(self):
        return self._value_out.flatten()


def env_creator(args):
    env = CustomEnvironment()
    #env = ss.color_reduction_v0(env, mode="B")
    #env = ss.dtype_v0(env, "float32")
    #env = ss.resize_v1(env, x_size=84, y_size=84)
    #env = ss.normalize_obs_v0(env, env_min=0, env_max=1)
    #env = ss.frame_stack_v1(env, 3)
    return env


if __name__ == "__main__":
    ray.init()

    env_name = "CustomEnvironment"

    register_env(env_name, lambda config: ParallelPettingZooEnv(env_creator(config)))
    ModelCatalog.register_custom_model("CNNModelV2", CNNModelV2)

    config = (
    PPOConfig()
    .environment(env=env_name, clip_actions=True)
    .env_runners(num_env_runners=4, rollout_fragment_length=128)
    .training(
        train_batch_size=512,
        lr=2e-5,
        gamma=0.99,
        lambda_=0.9,
        use_gae=True,
        clip_param=0.4,
        grad_clip=None,
        entropy_coeff=0.1,
        vf_loss_coeff=0.25,
    )
    .framework("torch")
    .debugging(log_level="ERROR")
    .resources(num_gpus=int(os.environ.get("RLLIB_NUM_GPUS", "0")))
    .update_from_dict({
        "num_sgd_iter": 10,
        "sgd_minibatch_size": 64,  # ✅ correct place
    })
)

    tune.run(
        "PPO",
        name="PPO",
        stop={"timesteps_total": 5000000 if not os.environ.get("CI") else 50000},
        checkpoint_freq=10,
        storage_path="~/ray_results/" + env_name,
        config=config.to_dict(),
    )